# Двумерная карта функции потерь: яркость адаптации La и яркость фона L_bg

### Аналитическое обоснование
В модели JND суммарная функция потерь складывается из ошибки восприятия пикселей изображения и ошибки восприятия фонового пространства экрана:
$$\text{Loss}(L_a, L_{bg}) = w_{\text{patch}} \cdot \text{Loss}_{\text{patch}}(L_a) + w_{bg} \cdot \text{Loss}_{bg}(L_a, L_{bg})$$

1. Компонента изображения $\text{Loss}_{\text{patch}}(L_a)$ зависит только от $L_a$ и достигает своего минимума при $L_a = L_a^{\text{patch}}$.
2. Компонента фона:
   $$\text{Loss}_{bg}(L_a, L_{bg}) = \left| \ln \frac{D(L_{bg}, L_a)}{D(L_a, L_{bg})} \right| \ge 0$$
   Она равна строго нулю тогда и только тогда, когда $L_a = L_{bg}$ (поскольку $D(L, L) / D(L, L) = 1$, а $\ln(1) = 0$).
3. Так как оба слагаемых неотрицательны, а их минимумы достигаются независимо:
   $$\min_{L_a, L_{bg}} \text{Loss}(L_a, L_{bg}) = w_{\text{patch}} \cdot \min_{L_a} \text{Loss}_{\text{patch}}(L_a) + w_{bg} \cdot 0$$
   Равенство достигается в единственной точке:
   $$L_a = L_a^{\text{patch}} \quad \text{и} \quad L_{bg} = L_a \implies \mathbf{L_{bg}^{\text{opt}} = L_a^* = L_a^{\text{patch}}}$$

In [1]:
import os
import glob
import math
from pathlib import Path
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import norm
from IPython.display import display

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'plotly_mimetype+notebook_connected'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
class ImageConverter:
    def __init__(self, L_MIN: float = 0.1, L_MAX: float = 300.0):
        self.M = np.array([
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041]
        ], dtype=np.float64)
        self.M_inv = np.array([
            [ 3.1338561, -1.6168667, -0.4906146],
            [-0.9787684,  1.9161415,  0.0334540],
            [ 0.0719453, -0.2289914,  1.4052427]
        ], dtype=np.float64)

        self.L_MAX = float(L_MAX)
        self.L_MIN = float(L_MIN)
        self.coord_white = np.array([0.3127, 0.3290, 0.3583], dtype=np.float64)
        self.eps = 1e-10

    def sRGB_Companding(self, linear_image: np.ndarray) -> np.ndarray:
        linear = np.clip(np.asarray(linear_image, dtype=np.float64), 0.0, 1.0)
        encoded = np.where(
            linear <= 0.0031308,
            linear * 12.92,
            1.055 * (linear ** (1.0 / 2.4)) - 0.055
        )
        return encoded

    def Inverse_sRGB_Companding(self, image: np.ndarray) -> np.ndarray:
        img = np.asarray(image, dtype=np.float64)
        linear = np.where(
            img <= 0.04045,
            img / 12.92,
            ((img + 0.055) / 1.055) ** 2.4
        )
        return linear

    def Linear_RGB_to_XYZ(self, image: np.ndarray) -> np.ndarray:
        XYZ_image = image.reshape(-1, 3)
        return (XYZ_image @ self.M.T).reshape(image.shape)

    def XYZ_to_xyzL(self, XYZ_image: np.ndarray) -> np.ndarray:
        sum_xyz = XYZ_image.sum(keepdims=True, axis=-1)
        xyz = XYZ_image / (sum_xyz + self.eps)

        is_black = (sum_xyz < self.eps).squeeze(-1)
        if np.any(is_black):
            xyz[is_black] = self.coord_white

        Y = XYZ_image[..., 1]
        L_norm = np.clip(Y, 0.0, 1.0)
        L_physical = self.L_MIN + L_norm * (self.L_MAX - self.L_MIN)
        L_physical = np.expand_dims(L_physical, axis=-1)

        xyzL = np.concatenate([xyz, L_physical], axis=-1)
        return xyzL

    def RGB_to_xyzL(self, image: np.ndarray) -> np.ndarray:
        norm_img = np.asarray(image, dtype=np.float64) / 255.0
        linRGBimg = self.Inverse_sRGB_Companding(norm_img)
        XYZ_image = self.Linear_RGB_to_XYZ(linRGBimg)
        xyzL_image = self.XYZ_to_xyzL(XYZ_image)
        return xyzL_image

    def L_to_gray(self, L: np.ndarray | float) -> np.ndarray | np.uint8:
        Y_lin = (L - self.L_MIN) / (self.L_MAX - self.L_MIN)
        Y_lin = np.clip(Y_lin, 0.0, 1.0)
        V = self.sRGB_Companding(Y_lin)
        gray_matrix = np.rint(V * 255).astype(np.uint8)
        return gray_matrix

In [3]:
@dataclass(frozen=True)
class AdaptationResult:
    La: float
    loss: float
    success: bool
    iterations: int
    function_evaluations: int
    message: str


def compute_screen_geometry(
    patch_shape: tuple[int, int],
    mode: str,
    distance_m: float = 0.5,
    screen_diag_inch: float = 16.0,
    screen_width_px: int = 1920,
    screen_height_px: int = 1080,
) -> tuple[float, int, int, int]:
    h_patch, w_patch = patch_shape
    n_patch = h_patch * w_patch

    diag_px = math.sqrt(screen_width_px ** 2 + screen_height_px ** 2)
    ppi = diag_px / screen_diag_inch
    pixel_size_m = 0.0254 / ppi
    phi_screen = pixel_size_m / distance_m

    n_total = screen_width_px * screen_height_px

    if mode == 'full':
        return phi_screen, n_patch, n_patch, 0

    if mode == 'orig':
        n_patch_screen = n_patch
        n_bg = max(0, n_total - n_patch_screen)
        return phi_screen, n_total, n_patch_screen, n_bg

    if mode == 'scale_15cm':
        target_size_m = 0.15
        target_max_px = int(np.round((target_size_m / 0.0254) * ppi))
        scale = target_max_px / max(h_patch, w_patch)
        target_w_px = int(np.round(w_patch * scale))
        target_h_px = int(np.round(h_patch * scale))
        n_patch_screen = target_w_px * target_h_px
        n_bg = max(0, n_total - n_patch_screen)
        phi = scale * phi_screen
        return phi, n_total, n_patch_screen, n_bg

    if mode == 'scale_8cm':
        target_size_m = 0.08
        target_max_px = int(np.round((target_size_m / 0.0254) * ppi))
        scale = target_max_px / max(h_patch, w_patch)
        target_w_px = int(np.round(w_patch * scale))
        target_h_px = int(np.round(h_patch * scale))
        n_patch_screen = target_w_px * target_h_px
        n_bg = max(0, n_total - n_patch_screen)
        phi = scale * phi_screen
        return phi, n_total, n_patch_screen, n_bg

    if mode == 'scale_50pct':
        n_patch_screen = n_total // 2
        n_bg = n_total - n_patch_screen
        scale = math.sqrt(n_patch_screen / n_patch)
        phi = scale * phi_screen
        return phi, n_total, n_patch_screen, n_bg

    raise ValueError(f"Неизвестный режим: {mode}")


class HistogramAdaptationJNDModel:
    def __init__(
        self,
        *,
        exposure_time: float = 1.0,
        angular_size_rad: float = 1e-3,
        detection_probability: float = 0.75,
        absolute_threshold: float = 1e-6,
        luminance_floor: float = 1e-6,
        l_min: float = 0.1,
        l_max: float = 300.0,
        screen_width_px: int = 1920,
        screen_height_px: int = 1080,
        l_gray: float = 60.0
    ) -> None:
        self.t = float(exposure_time)
        self.phi_d = float(angular_size_rad)
        self.p = float(detection_probability)
        self.L0 = float(absolute_threshold)
        self.luminance_floor = float(luminance_floor)

        self.l_min = float(l_min)
        self.l_max = float(l_max)
        self.screen_width_px = int(screen_width_px)
        self.screen_height_px = int(screen_height_px)
        self.l_gray = float(l_gray)

        self.la = self.l_gray
        self.bounds = np.empty(0, dtype=np.float64)
        self.len_left = 0

    @staticmethod
    def _K(lambd, b1=0.98, b2=0.1, lambd1=2e-2):
        lambd = np.asarray(lambd, dtype=np.float64)
        return np.where(lambd <= 1.0, b1 * (1.0 + lambd1 / lambd), lambd**b2)

    @staticmethod
    def _A(La, a1=4.8e6, a2=7.1e3, a3=4.5e-4):
        return 1.0 + np.sqrt(a1 * La) + a2 * La * (1.0 + np.sqrt(a3 * La))

    def _Lambda(self, lambd):
        return lambd * self._K(lambd)

    @staticmethod
    def _eta(La, L3=1e6, L4=1.0, a4=600.0):
        return 1.0 + L3 / (1.0 + a4 * La * (L4 + La))

    def _C(self, La):
        tau_min = 0.05
        T = self.t / (tau_min * self._eta(La))
        return 1.0 + 1.0 / T

    @staticmethod
    def _func_phi_1(La, c2=0.25, L1=10.0, L2=2.6e-5):
        return (1.0 + L1 / (L2 + La)) ** c2

    def _S(self, La, theta_1=6.1, a_d=2.0, inf_phi_1=1.0):
        phi_0_min = 0.5 * (1.0 / 60.0) * (np.pi / 180.0)
        phi_0 = self._func_phi_1(La) * phi_0_min / inf_phi_1
        return (theta_1 * phi_0 / self.phi_d + 1.0) ** a_d

    def _l(self, La):
        return self.L0 * self._A(La)

    def _P(self, p_ref=0.75):
        return abs(norm.ppf(self.p) / norm.ppf(p_ref))

    def D(self, L, La):
        L = np.maximum(np.asarray(L, dtype=np.float64), self.luminance_floor)
        La = np.maximum(np.asarray(La, dtype=np.float64), self.luminance_floor)
        lambd = L / (1e2 * self._l(La))
        return self.L0 * self._A(La) * self._C(La) * self._Lambda(lambd) * self._S(La) * self._P()

    def prepare_histogram(self, luminance, probability) -> tuple[np.ndarray, np.ndarray]:
        L = np.asarray(luminance, dtype=np.float64).reshape(-1)
        p = np.asarray(probability, dtype=np.float64).reshape(-1)

        L = np.maximum(L, self.luminance_floor)
        keep = p > 0
        L, p = L[keep], p[keep]

        unique_L, inverse = np.unique(L, return_inverse=True)
        merged_p = np.zeros(unique_L.size, dtype=np.float64)
        np.add.at(merged_p, inverse, p)
        merged_p /= merged_p.sum()
        return unique_L, merged_p

    def extract_image_histogram(
        self,
        patch: np.ndarray,
        n_total: int,
        n_patch_screen: int,
        n_bg: int,
        l_bg: float | None = None
    ) -> tuple[np.ndarray, np.ndarray]:
        flat = np.asarray(patch, dtype=np.float64).ravel()
        bg_lum = self.l_gray if l_bg is None else float(l_bg)

        unique_L, counts = np.unique(flat, return_counts=True)
        probs = (counts / flat.size) * (n_patch_screen / n_total)

        if n_bg > 0:
            unique_L = np.append(unique_L, bg_lum)
            probs = np.append(probs, n_bg / n_total)

        return self.prepare_histogram(unique_L, probs)

    def loss(self, La: float, luminance, probability) -> float:
        if not np.isfinite(La) or La <= 0:
            return math.inf
        L, p = self.prepare_histogram(luminance, probability)
        ratio = self.D(L, La) / self.D(La, L)
        if np.any(~np.isfinite(ratio)) or np.any(ratio <= 0):
            return math.inf
        return float(np.sum(p * np.abs(np.log(ratio))))

    def fit(self, luminance, probability, bounds: tuple[float, float] | None = None, xatol: float = 1e-6) -> AdaptationResult:
        L, p = self.prepare_histogram(luminance, probability)
        if bounds is None and L.size == 1:
            center = float(L[0])
            lower, upper = max(center / 100.0, self.luminance_floor), center * 100.0
        else:
            lower, upper = bounds if bounds is not None else (float(L.min()), float(L.max()))
        lower = max(float(lower), self.luminance_floor)
        upper = float(upper)
        if lower >= upper:
            lower = max(lower / 2.0, self.luminance_floor)
            upper = upper * 2.0

        def objective(log_La):
            La = math.exp(float(log_La))
            ratio = self.D(L, La) / self.D(La, L)
            return float(np.sum(p * np.abs(np.log(ratio))))

        opt = minimize_scalar(
            objective,
            bounds=(math.log(lower), math.log(upper)),
            method="bounded",
            options={"xatol": float(xatol)},
        )
        La = math.exp(float(opt.x))
        return AdaptationResult(
            La=La,
            loss=float(opt.fun),
            success=bool(opt.success),
            iterations=int(opt.nit),
            function_evaluations=int(opt.nfev),
            message=str(opt.message),
        )

    def find_la(self, patch: np.ndarray, n_total: int, n_patch_screen: int, n_bg: int, l_bg: float | None = None) -> float:
        L, p = self.extract_image_histogram(patch, n_total, n_patch_screen, n_bg, l_bg)
        res = self.fit(L, p)
        self.la = res.La
        return self.la

    def build_level_boundaries(self) -> np.ndarray:
        right_bounds: list[float] = []
        curr_l = self.la
        while curr_l < self.l_max:
            step = float(self.D(L=curr_l, La=self.la))
            if step <= 0:
                break
            curr_l += step
            right_bounds.append(min(curr_l, self.l_max))

        left_bounds: list[float] = []
        curr_l = self.la
        while curr_l > self.L0:
            step = float(self.D(L=curr_l, La=self.la))
            if step <= 0:
                break
            curr_l -= step
            left_bounds.append(max(curr_l, self.L0))

        self.len_left = len(left_bounds)
        left_bounds.reverse()
        self.bounds = np.array(left_bounds + [self.la] + right_bounds, dtype=np.float64)
        return self.bounds

    def l_to_k(self, l_val: float | np.ndarray) -> int | np.ndarray:
        idx = np.searchsorted(self.bounds, l_val, side="right") - 1
        idx = np.clip(idx, 0, len(self.bounds) - 1)
        return idx - self.len_left


In [4]:
def plot_interactive_loss_surface(
    img_path,
    mode: str = 'scale_15cm',
    grid_size: int = 50,
    stride: int = 4
):
    img_path = Path(img_path)
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        print(f"Ошибка загрузки: {img_path}")
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    converter = ImageConverter(L_MIN=0.1, L_MAX=300.0)
    l_physical = converter.RGB_to_xyzL(img_rgb)[..., 3]
    h, w = l_physical.shape
    mean_lum = float(np.mean(l_physical))
    
    phi, n_total, n_patch_screen, n_bg = compute_screen_geometry(
        patch_shape=(h, w),
        mode=mode
    )
    
    model = HistogramAdaptationJNDModel(angular_size_rad=phi)
    
    l_sub = l_physical[::stride, ::stride].ravel()
    u_patch, counts_p = np.unique(l_sub, return_counts=True)
    p_patch = counts_p / l_sub.size
    
    w_p = n_patch_screen / n_total
    w_bg = n_bg / n_total
    
    la_grid = np.geomspace(0.1, 300.0, grid_size)
    lbg_grid = np.geomspace(0.1, 300.0, grid_size)
    
    loss_patch_1d = np.zeros(grid_size)
    for i, la in enumerate(la_grid):
        d_p_la = model.D(u_patch, la)
        d_la_p = model.D(la, u_patch)
        ratio = d_p_la / d_la_p
        loss_patch_1d[i] = float(np.sum(p_patch * np.abs(np.log(ratio))))
        
    idx_p_opt = int(np.argmin(loss_patch_1d))
    la_patch_opt = float(la_grid[idx_p_opt])
    
    loss_bg_2d = np.zeros((grid_size, grid_size))
    for i, lbg in enumerate(lbg_grid):
        for j, la in enumerate(la_grid):
            r = model.D(lbg, la) / model.D(la, lbg)
            loss_bg_2d[i, j] = abs(math.log(r))
            
    loss_2d = w_p * loss_patch_1d[np.newaxis, :] + w_bg * loss_bg_2d
    
    min_idx_2d = np.unravel_index(np.argmin(loss_2d), loss_2d.shape)
    bg_opt = float(lbg_grid[min_idx_2d[0]])
    la_opt = float(la_grid[min_idx_2d[1]])
    min_loss_val = float(loss_2d[min_idx_2d])
    
    LA_mesh, LBG_mesh = np.meshgrid(la_grid, lbg_grid)
    customdata = np.stack([LA_mesh, LBG_mesh], axis=-1)
    
    surf = go.Surface(
        x=np.log10(LA_mesh),
        y=np.log10(LBG_mesh),
        z=loss_2d,
        customdata=customdata,
        hovertemplate='La: %{customdata[0]:.2f} кд/м²<br>L_bg: %{customdata[1]:.2f} кд/м²<br>Loss: %{z:.4f}<extra></extra>',
        colorscale='Viridis',
        colorbar=dict(title='Loss')
    )
    
    diag_z = [loss_2d[k, k] for k in range(grid_size)]
    line_diag = go.Scatter3d(
        x=np.log10(la_grid),
        y=np.log10(la_grid),
        z=diag_z,
        mode='lines',
        line=dict(color='white', width=5),
        name='Линия La = L_bg'
    )
    
    pt_opt = go.Scatter3d(
        x=[np.log10(la_opt)],
        y=[np.log10(bg_opt)],
        z=[min_loss_val],
        mode='markers+text',
        marker=dict(size=8, color='red', symbol='diamond'),
        text=[f'Минимум (La={la_opt:.1f}, Lbg={bg_opt:.1f})'],
        textposition='top center',
        name='Глобальный минимум'
    )
    
    fig = go.Figure(data=[surf, line_diag, pt_opt])
    
    tick_vals = [-1, 0, 1, 2, math.log10(300.0)]
    tick_text = ['0.1', '1', '10', '100', '300']
    
    fig.update_layout(
        title=f"{img_path.name} | Масштаб 15 см (патч {int(w_p*100)}% экрана) | Оптимум La = L_bg = {la_opt:.2f} кд/м²",
        scene=dict(
            xaxis=dict(
                title='Яркость адаптации La (кд/м²)',
                tickvals=tick_vals,
                ticktext=tick_text
            ),
            yaxis=dict(
                title='Яркость фона L_bg (кд/м²)',
                tickvals=tick_vals,
                ticktext=tick_text
            ),
            zaxis=dict(title='Loss'),
            camera=dict(
                eye=dict(x=-1.5, y=-1.5, z=1.2)
            )
        ),
        width=950,
        height=680,
        margin=dict(l=10, r=10, b=10, t=50)
    )
    
    fig.show()
    
    diff = abs(bg_opt - la_patch_opt)
    matches = (diff < 1e-2)
    print(f"Изображение: {img_path.name} | Средняя яркость: {mean_lum:.2f} кд/м²")
    print(f"• Оптимум патча без фона: La = {la_patch_opt:.2f} кд/м²")
    print(f"• Глобальный 2D минимум: L_bg = {bg_opt:.2f} кд/м², La = {la_opt:.2f} кд/м²")
    print(f"• Совпадение L_bg_opt == La_patch_opt: {matches} (разница: {diff:.4f} кд/м²)")
    print(f"• Минимальный лосс: {min_loss_val:.6f}\n")
    
    return {
        'Файл': img_path.name,
        'Средняя яркость (кд/м²)': mean_lum,
        'La патча (кд/м²)': la_patch_opt,
        'Оптимальный L_bg (кд/м²)': bg_opt,
        'Глобальный La* (кд/м²)': la_opt,
        'Совпадение L_bg==La': matches,
        'Минимальный Loss': min_loss_val
    }

In [5]:
image_paths = [f'../examples/{i}.jpg' for i in range(1, 6)]
if not os.path.exists(image_paths[0]):
    image_paths = [f'examples/{i}.jpg' for i in range(1, 6)]

results_summary = []
for p in image_paths:
    res = plot_interactive_loss_surface(p, mode='scale_15cm', grid_size=50)
    if res:
        results_summary.append(res)

Изображение: 1.jpg | Средняя яркость: 108.73 кд/м²
• Оптимум патча без фона: La = 112.55 кд/м²
• Глобальный 2D минимум: L_bg = 112.55 кд/м², La = 112.55 кд/м²
• Совпадение L_bg_opt == La_patch_opt: True (разница: 0.0000 кд/м²)
• Минимальный лосс: 0.143259



Изображение: 2.jpg | Средняя яркость: 21.47 кд/м²
• Оптимум патча без фона: La = 9.70 кд/м²
• Глобальный 2D минимум: L_bg = 9.70 кд/м², La = 9.70 кд/м²
• Совпадение L_bg_opt == La_patch_opt: True (разница: 0.0000 кд/м²)
• Минимальный лосс: 0.392410



Изображение: 3.jpg | Средняя яркость: 14.53 кд/м²
• Оптимум патча без фона: La = 8.24 кд/м²
• Глобальный 2D минимум: L_bg = 8.24 кд/м², La = 8.24 кд/м²
• Совпадение L_bg_opt == La_patch_opt: True (разница: 0.0000 кд/м²)
• Минимальный лосс: 0.553255



Изображение: 4.jpg | Средняя яркость: 100.38 кд/м²
• Оптимум патча без фона: La = 68.94 кд/м²
• Глобальный 2D минимум: L_bg = 68.94 кд/м², La = 68.94 кд/м²
• Совпадение L_bg_opt == La_patch_opt: True (разница: 0.0000 кд/м²)
• Минимальный лосс: 0.230703



Изображение: 5.jpg | Средняя яркость: 9.16 кд/м²
• Оптимум патча без фона: La = 0.60 кд/м²
• Глобальный 2D минимум: L_bg = 0.60 кд/м², La = 0.60 кд/м²
• Совпадение L_bg_opt == La_patch_opt: True (разница: 0.0000 кд/м²)
• Минимальный лосс: 0.594275



In [6]:
df_res = pd.DataFrame(results_summary)
display(df_res)

,Файл,Средняя яркость (кд/м²),La патча (кд/м²),Оптимальный L_bg (кд/м²),Глобальный La* (кд/м²),Совпадение L_bg==La,Минимальный Loss
0,1.jpg,108.7321,112.5515,112.5515,112.5515,True,0.1433
1,2.jpg,21.4713,9.7035,9.7035,9.7035,True,0.3924
2,3.jpg,14.5265,8.2407,8.2407,8.2407,True,0.5533
3,4.jpg,100.3806,68.9392,68.9392,68.9392,True,0.2307
4,5.jpg,9.1554,0.6034,0.6034,0.6034,True,0.5943
